# Task 3 - Weather Scraping with Fallback

The goal is to scrape weather data from Timeanddate first.
If the first website fails, the code should handle the error and try a second weather website.
The result will be saved locally in a text file.


In [1]:
import re
import requests
from bs4 import BeautifulSoup

In [2]:
timeanddate_url = "https://www.timeanddate.com/weather/finland/helsinki"
wunderground_url = "https://www.wunderground.com/weather/fi/helsinki"

weather_result = ""

try:
    # First attempt: Timeanddate
    response = requests.get(timeanddate_url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    weather_result = (
        "Source: Timeanddate\n"
        "Location: Helsinki, Finland\n"
        "Weather page retrieved successfully."
    )

except requests.RequestException as error:
    print("Timeanddate failed:", error)
    print("Trying Weather Underground...")

    # Fallback: Weather Underground
    response = requests.get(wunderground_url, timeout=10)
    response.raise_for_status()

    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    page_text = soup.get_text(" ", strip=True)

    # Search the current-weather section
    updated_position = page_text.find("Updated")

    if updated_position != -1:
        weather_text = page_text[updated_position:updated_position + 600]
    else:
        weather_text = page_text

    # Match temperature, feels-like temperature and condition together
    current_weather_match = re.search(
        r"(-?\d+(?:\.\d+)?)\s*°?\s*([FC])?\s+like\s+"
        r"(-?\d+(?:\.\d+)?)\s*°?\s+"
        r"([A-Za-z ]+?)\s+[NSEW]{1,3}\s+\d+",
        weather_text,
        re.IGNORECASE
    )

    if current_weather_match:
        temperature = float(current_weather_match.group(1))
        unit = current_weather_match.group(2)
        feels_like = float(current_weather_match.group(3))
        condition = current_weather_match.group(4).strip()

        # Convert only when Weather Underground explicitly says Fahrenheit
        if unit and unit.upper() == "F":
            temperature = round((temperature - 32) * 5 / 9, 1)
            feels_like = round((feels_like - 32) * 5 / 9, 1)

        weather_result = (
            "Source: Weather Underground\n"
            "Location: Helsinki, Finland\n"
            f"Temperature: {temperature:g} °C\n"
            f"Feels like: {feels_like:g} °C\n"
            f"Condition: {condition}"
        )

    else:
        weather_result = (
            "Source: Weather Underground\n"
            "Location: Helsinki, Finland\n"
            "Weather data could not be extracted."
        )

print(weather_result)

Timeanddate failed: 403 Client Error: Forbidden for url: https://www.timeanddate.com/weather/finland/helsinki
Trying Weather Underground...
Source: Weather Underground
Location: Helsinki, Finland
Temperature: 5.6 °C
Feels like: 3.9 °C
Condition: Partly Cloudy


In [3]:
# Save Task 3 weather result to a local text file

output_path = "../output/task3_weather.txt"

with open(output_path, "w", encoding="utf-8") as file:
    file.write(weather_result)

print("Task 3 weather data saved successfully.")

Task 3 weather data saved successfully.
